# 01_centralized_baseline.ipynb

Este notebook implementa el escenario base de entrenamiento centralizado utilizado como referencia para el resto de experimentos del proyecto. Sobre el dataset CIFAR-10 se entrena una red convolucional simple (SimpleCNN) utilizando la totalidad de los datos en un único nodo, sin aplicar técnicas de aprendizaje federado.

Además del entrenamiento y evaluación del modelo, se integra CodeCarbon para capturar métricas de sostenibilidad, incluyendo tiempo de ejecución, consumo energético y emisiones operativas de carbono. Al finalizar el experimento, los resultados de rendimiento y las métricas ambientales se almacenan en un fichero CSV estructurado, que servirá como punto de comparación para los posteriores escenarios federados y para el cálculo futuro del Software Carbon Intensity (SCI).

In [1]:
DATA_DIR = "../data"

DATASET = "CIFAR10"

SCENARIO = "centralized"

BATCH_SIZE = 64

EPOCHS = 5

LEARNING_RATE = 0.001

RANDOM_SEED = 42


1. Carga del Dataset y DataLoader

In [2]:

import torch

import torch.nn as nn

from torch.utils.data import DataLoader

from torchvision import datasets
from torchvision import transforms

transform = transforms.ToTensor()

train_dataset = datasets.CIFAR10(
    root="../data",
    train=True,
    download=True,
    transform=transform
)

test_dataset = datasets.CIFAR10(
    root="../data",
    train=False,
    download=True,
    transform=transform
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)


2. Modelo

In [3]:
import torch.nn as nn

class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 8 * 8, 128),
            nn.ReLU(),
            nn.Linear(128, 10)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

In [4]:
model = SimpleCNN()

print(model)

SimpleCNN(
  (features): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=4096, out_features=128, bias=True)
    (2): ReLU()
    (3): Linear(in_features=128, out_features=10, bias=True)
  )
)


3. Función de pérdida y optimización

In [5]:
import torch.optim as optim
import torch.nn as nn
criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE
)

In [6]:
images, labels = next(iter(train_loader))

outputs = model(images)

print(outputs.shape)

torch.Size([64, 10])


4. Función de entrenamiento

In [7]:
def train_model(
    model,
    train_loader,
    optimizer,
    criterion,
    epochs
):

    model.train()

    for epoch in range(epochs):

        running_loss = 0.0

        for images, labels in train_loader:

            optimizer.zero_grad()

            outputs = model(images)

            loss = criterion(outputs, labels)

            loss.backward()

            optimizer.step()

            running_loss += loss.item()

        epoch_loss = running_loss / len(train_loader)

        print(
            f"Epoch {epoch + 1}/{epochs} - "
            f"Loss: {epoch_loss:.4f}"
        )

Función de evaluación

In [8]:
def evaluate_model(
    model,
    test_loader
):

    model.eval()

    correct = 0
    total = 0

    with torch.no_grad():

        for images, labels in test_loader:

            outputs = model(images)

            _, predicted = torch.max(outputs, 1)

            total += labels.size(0)

            correct += (
                predicted == labels
            ).sum().item()

    accuracy = correct / total

    return accuracy

Función principal

arranca CodeCarbon,
entrena,
evalúa,
crea la fila del dataset,
la guarda en CSV.

In [ ]:
import time
from pathlib import Path
import pandas as pd

from codecarbon import EmissionsTracker

def run_experiment(
    scenario,
    model,
    train_loader,
    test_loader,
    optimizer,
    criterion,
    epochs
):

    tracker = EmissionsTracker(save_to_file=False)

    start_time = time.time()

    tracker.start()

    tracker.start_task("training")

    train_model(
        model,
        train_loader,
        optimizer,
        criterion,
        epochs
    )

    training_data = tracker.stop_task()

    emissions = tracker.stop()

    training_time = (
        time.time() - start_time
    )

    accuracy = evaluate_model(
        model,
        test_loader
    )

    result = {
    
    # ==================================================
    # Identificación (anonimizada) de la máquina y del experimento
    # ==================================================

    "experiment_id": time.strftime("%Y-%m-%d_%H-%M-%S"),
    "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),

    # ==================================================
    # Escenario experimental
    # ==================================================

    "scenario": scenario,
    "record_type": "experiment",

    # ==================================================
    # Dataset y modelo
    # ==================================================

    "dataset": DATASET,
    "model": "SimpleCNN",

    # ==================================================
    # Configuración de entrenamiento
    # ==================================================

    "epochs": epochs,
    "batch_size": BATCH_SIZE,
    "learning_rate": LEARNING_RATE,
    "random_seed": RANDOM_SEED,

    # ==================================================
    # Configuración federada
    # ==================================================

    "num_clients": None,
    "client_id": None,
    "round": None,

    # ==================================================
    # Resultados del modelo
    # ==================================================

    "accuracy": accuracy,

    # ==================================================
    # Tiempo
    # ==================================================

    "training_time_s": training_time,
    "duration_s": training_data.duration,

    # ==================================================
    # Energía
    # ==================================================

    "energy_kwh": training_data.energy_consumed,
    "cpu_energy_kwh": training_data.cpu_energy,
    "ram_energy_kwh": training_data.ram_energy,

    # ==================================================
    # Emisiones
    # ==================================================

    "operational_emissions_kg": training_data.emissions,
    "carbon_intensity": None,

        
    # ==================================================
    # SCI (Software Carbon Intensity)
    # ==================================================
    
    "embedded_carbon_kg": None,
    "sci": None,
}

    results_file = "../results/experiments.csv"

    df = pd.DataFrame([result])

    if Path(results_file).exists():

        df.to_csv(
            results_file,
            mode="a",
            header=False,
            index=False
        )

    else:

        df.to_csv(
            results_file,
            index=False
        )

    return result

Ejecución del experimento:

In [10]:
import pandas as pd
from pathlib import Path
result = run_experiment(
    scenario=SCENARIO,
    model=model,
    train_loader=train_loader,
    test_loader=test_loader,
    optimizer=optimizer,
    criterion=criterion,
    epochs=EPOCHS
)

print(result)

[codecarbon WARNING @ 08:57:29] Multiple instances of codecarbon are allowed to run at the same time.
[codecarbon INFO @ 08:57:29] [setup] RAM Tracking...
[codecarbon INFO @ 08:57:29] [setup] CPU Tracking...
[codecarbon INFO @ 08:57:29] [setup] GPU Tracking...
[codecarbon INFO @ 08:57:29] No GPU found.
[codecarbon WARNING @ 08:57:30] We saw that you have a 12th Gen Intel(R) Core(TM) i7-1255U but we don't know it. Please contact us.
[codecarbon WARNING @ 08:57:30] We will use the default power consumption of 4 W per thread for your 12 CPU, so 48W.
[codecarbon WARNING @ 08:57:30] No CPU tracking mode found. Falling back on estimation based on TDP for CPU. 
 Linux OS detected: Please ensure RAPL files exist, and are readable, at /sys/class/powercap/intel-rapl/subsystem to measure CPU

[codecarbon INFO @ 08:57:30] CPU Model on constant consumption mode: 12th Gen Intel(R) Core(TM) i7-1255U
[codecarbon WARNING @ 08:57:30] No CPU tracking mode found. Falling back on CPU load mode.
[codecarbon

Epoch 1/5 - Loss: 1.4893
Epoch 2/5 - Loss: 1.1282
Epoch 3/5 - Loss: 0.9678
Epoch 4/5 - Loss: 0.8572


[codecarbon INFO @ 09:00:38] Energy consumed for RAM : 0.000478 kWh. RAM Power : 10.0 W
[codecarbon INFO @ 09:00:38] Delta energy consumed for CPU with cpu_load : 0.000493 kWh, power : 10.321752095860466 W
[codecarbon INFO @ 09:00:38] Energy consumed for All CPU : 0.000493 kWh
[codecarbon INFO @ 09:00:38] 0.000971 kWh of electricity and 0.000000 L of water were used since the beginning.


Epoch 5/5 - Loss: 0.7810
{'experiment_id': '2026-10-04_09-00-41', 'timestamp': '2026-10-04 09:00:41', 'scenario': 'centralized', 'record_type': 'experiment', 'dataset': 'CIFAR10', 'model': 'SimpleCNN', 'epochs': 5, 'batch_size': 64, 'learning_rate': 0.001, 'random_seed': 42, 'num_clients': None, 'client_id': None, 'round': None, 'accuracy': 0.6869, 'training_time_s': 188.9929530620575, 'duration_s': 168.82182723200003, 'energy_kwh': 0.000970406525115403, 'cpu_energy_kwh': 0.0004928872083431818, 'ram_energy_kwh': 0.0004775193167722212, 'operational_emissions_kg': 0.0001688992556963359, 'carbon_intensity': None}


In [11]:

import pandas as pd
from pathlib import Path

results = pd.read_csv(
    "../results/experiments.csv"
)

results

,experiment_id,timestamp,scenario,record_type,dataset,model,epochs,batch_size,learning_rate,random_seed,...,client_id,round,accuracy,training_time_s,duration_s,energy_kwh,cpu_energy_kwh,ram_energy_kwh,operational_emissions_kg,carbon_intensity
0,2026-10-04_09-00-41,2026-10-04 09:00:41,centralized,experiment,CIFAR10,SimpleCNN,5,64,0.001,42,...,NaN,NaN,0.6869,188.992953,168.821827,0.00097,0.000493,0.000478,0.000169,NaN
